In [1]:
from openai import OpenAI
from qdrant_client import QdrantClient

In [8]:
openai = OpenAI()

In [2]:
model = "text-embedding-3-small"
collection_name="Amazon-shopping-collection-01"
qdrant_client = QdrantClient(url="http://localhost:6333")

In [10]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding

## RAG > Retrieval Function

In [11]:
def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(collection_name=collection_name, query=query_embedding, limit=k)
    context_ids = []
    context = []
    scores = []
    ratings = []

    print(results)
    for pt in results.points:
        context_ids.append(pt.payload['parent_asin'])
        context.append(pt.payload['preprocessed_description'])
        scores.append(pt.score)
        ratings.append(pt.payload['average_rating'])
        
    return {
        'retrieved_context_ids': context_ids,
        'retrieved_context': context,
        'similarity_scores': scores,
        'retrieved_context_ratings': ratings
    }

In [12]:
retrieve_context = retrieve_data("Ethernet Cable?", k=10)
retrieve_context

points=[ScoredPoint(id=25, version=1, score=0.5263114, payload={'preprocessed_description': 'Cat 8 Ethernet Cable JUSTITUDE Nylon Braided High Speed LAN Network Cable with Clips 40Gbps 2000Mhz SFTP LAN Wires CAT8 Gold Plated RJ45 Connector Ethernet Cable (3.3ft/1m, Black)AMAZING SPEED： Cat 8 Ethernet Cable support bandwidth up to 2000MHz and up to 40Gbps data transmitting speed.Compared with existing cat5/cat6/Cat7 lan cable it is the fastest S/FTP Ethernet Cablethe that at speeds ,Cat8 lan cable meet the requirements for any efficient conditions.It could connect to LAN/WAN segments and networking gear at maximum speeds and surf the net, stream videos, music and other data at high speed without worrying about slow network Speed. ATTRACTIVE APPEARANCE AND VERIFIED NETWORK TEST: Braided nylon made the cat8 lan cable more elegant and stronger than others,at the same time,because of the special surface the cable is dirty proof!4Pairs 100% pure & thick Copper wires make sure the speed when 

{'retrieved_context_ids': ['B07Z9BMZ59',
  'B0BRC5NT7Y',
  'B0BXLBG8W3',
  'B012PSPRR0',
  'B0BYV5MDHC',
  'B08K4SVTRN',
  'B00APTQR62',
  'B094QQHJP7',
  'B08DRJ2M86',
  'B09FSZ5T32'],
 'retrieved_context': ['Cat 8 Ethernet Cable JUSTITUDE Nylon Braided High Speed LAN Network Cable with Clips 40Gbps 2000Mhz SFTP LAN Wires CAT8 Gold Plated RJ45 Connector Ethernet Cable (3.3ft/1m, Black)AMAZING SPEED： Cat 8 Ethernet Cable support bandwidth up to 2000MHz and up to 40Gbps data transmitting speed.Compared with existing cat5/cat6/Cat7 lan cable it is the fastest S/FTP Ethernet Cablethe that at speeds ,Cat8 lan cable meet the requirements for any efficient conditions.It could connect to LAN/WAN segments and networking gear at maximum speeds and surf the net, stream videos, music and other data at high speed without worrying about slow network Speed. ATTRACTIVE APPEARANCE AND VERIFIED NETWORK TEST: Braided nylon made the cat8 lan cable more elegant and stronger than others,at the same time,be

### Format retrieved context

In [57]:
def process_context(context):
    formated_context = ""
    for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
        formated_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formated_context

In [58]:
preprocessed_context = process_context(retrieve_context)
print(preprocessed_context)

- ID: B07Z9BMZ59, rating: 4.4, description: Cat 8 Ethernet Cable JUSTITUDE Nylon Braided High Speed LAN Network Cable with Clips 40Gbps 2000Mhz SFTP LAN Wires CAT8 Gold Plated RJ45 Connector Ethernet Cable (3.3ft/1m, Black)AMAZING SPEED： Cat 8 Ethernet Cable support bandwidth up to 2000MHz and up to 40Gbps data transmitting speed.Compared with existing cat5/cat6/Cat7 lan cable it is the fastest S/FTP Ethernet Cablethe that at speeds ,Cat8 lan cable meet the requirements for any efficient conditions.It could connect to LAN/WAN segments and networking gear at maximum speeds and surf the net, stream videos, music and other data at high speed without worrying about slow network Speed. ATTRACTIVE APPEARANCE AND VERIFIED NETWORK TEST: Braided nylon made the cat8 lan cable more elegant and stronger than others,at the same time,because of the special surface the cable is dirty proof!4Pairs 100% pure & thick Copper wires make sure the speed when you surf on net.Our Cat8 Ethernet Cable is verifi

## Prompt

In [32]:
def build_prompt(context, query):
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - answer the question based on the provided context only.
    - never use word context and refer to it as the available products.
    - do not use markdown formatting.

    Context:
    {context}

    Question:
    {query}
    """

    return prompt

In [59]:
prompt = build_prompt(preprocessed_context, 'Do you have Ethernet Cable ?')
print(prompt)


    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - answer the question based on the provided context only.
    - never use word context and refer to it as the available products.
    - do not use markdown formatting.

    Context:
    - ID: B07Z9BMZ59, rating: 4.4, description: Cat 8 Ethernet Cable JUSTITUDE Nylon Braided High Speed LAN Network Cable with Clips 40Gbps 2000Mhz SFTP LAN Wires CAT8 Gold Plated RJ45 Connector Ethernet Cable (3.3ft/1m, Black)AMAZING SPEED： Cat 8 Ethernet Cable support bandwidth up to 2000MHz and up to 40Gbps data transmitting speed.Compared with existing cat5/cat6/Cat7 lan cable it is the fastest S/FTP Ethernet Cablethe that at speeds ,Cat8 lan cable meet the requirements for any efficient conditions.It could connect to LAN/WAN segments and networking gear at maximum speeds and surf the net, stream videos, music and other data at high spee

In [43]:
model_name = "gpt-5.4-nano"
openai = OpenAI()

In [50]:

def generate_answer(prompt):
    
    response = openai.chat.completions.create(
            model=model_name,
            messages=[{"role":"system","content": prompt}]
                     )
    return response.choices[0].message.content

In [60]:
response = generate_answer(prompt)
response

'Yes. We have a Cat 8 Ethernet Cable (JUSTITUDE) with nylon braided construction and clips, supporting up to 40Gbps and 2000MHz, with gold-plated RJ45 connectors and S/FTP (SFTP/STP) shielding.'

### Combined RAG Pipeline

In [65]:
def rag_pipeline(question, top_k= 5):
    context = retrieve_data(question, top_k)
    preprocessed_context = process_context(context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)
    return answer

In [69]:
answer = rag_pipeline("give me top few DVD Drive?", 10);
print(answer)

points=[ScoredPoint(id=28, version=1, score=0.4768792, payload={'preprocessed_description': 'Original DVD Drive Replacement for Nintendo Wii Game Console Plug-and-Play Unit', 'image': 'https://m.media-amazon.com/images/I/51IbdwUfBLL._AC_.jpg', 'rating_number': 563, 'price': 29.41, 'average_rating': 4.7, 'parent_asin': 'B00APTQR62'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=10, version=1, score=0.43392196, payload={'preprocessed_description': "Gueray External CD DVD Drive for Laptop with USB 3.0 & Type-C Converter CD DVD Drive Portable CD DVD RW Reader Re-Writer Burner for Laptop Desktop PC Compatible with Windows Mac OS Vista Linux System (Rhombus shape)【High Speed via USB 3.0 & Type-C】Gueray external DVD drive supports the latest USB 3.0 technology to achieve the faster data transfer rate and more stable performance with the highest theoretical rate that is up to 5 GBPS. It is also compatible with USB 2.0 / 1.0, comes with Type-C converter, which is compatible wi